# Phase 1A: Raw Dataset Inspection

This notebook inspects raw CSV files in `datasets/raw/` and writes a summary report to `reports/raw_data_report.txt`.

**Important:** This notebook does not modify any data.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

def resolve_project_root() -> Path:
    cwd = Path.cwd()
    if (cwd / 'soil_ai_system').exists():
        return cwd / 'soil_ai_system'
    if cwd.name == 'notebooks':
        return cwd.parent
    return cwd

project_root = resolve_project_root()
sys.path.append(str(project_root))

from config import RAW_DATA_PATH, REPORT_PATH, RAW_DATA_REPORT_FILENAME, RAW_DATASETS

raw_dir = (project_root / RAW_DATA_PATH).resolve()
report_path = (project_root / REPORT_PATH / RAW_DATA_REPORT_FILENAME).resolve()

print(f'Raw data directory: {raw_dir}')
print(f'Report path: {report_path}')

In [ ]:
datasets = {}
missing = []

for name, filename in RAW_DATASETS.items():
    path = raw_dir / filename
    if path.exists():
        datasets[name] = pd.read_csv(path)
        print(f'Loaded {name}: {datasets[name].shape}')
    else:
        missing.append(str(path))

if missing:
    print('Missing datasets:')
    for item in missing:
        print(f'- {item}')

In [ ]:
def inspect_dataset(name: str, df: pd.DataFrame) -> dict:
    summary = {
        'shape': df.shape,
        'columns': list(df.columns),
        'dtypes': df.dtypes.astype(str).to_dict(),
        'null_counts': df.isnull().sum().to_dict(),
        'duplicate_rows': int(df.duplicated().sum()),
        'sample_rows': df.head(5).to_dict(orient='records')
    }
    return summary

summaries = {name: inspect_dataset(name, df) for name, df in datasets.items()}
summaries

In [ ]:
def normalize_column(col: str) -> str:
    return ''.join(ch for ch in col.lower() if ch.isalnum())

schema = {name: set(df.columns) for name, df in datasets.items()}
union_cols = set().union(*schema.values()) if schema else set()

missing_by_dataset = {name: sorted(list(union_cols - cols)) for name, cols in schema.items()}

normalized_map = {}
for name, cols in schema.items():
    normalized_map[name] = {normalize_column(col): col for col in cols}

potential_mismatches = []
if len(normalized_map) > 1:
    all_norm = set().union(*[set(m.keys()) for m in normalized_map.values()])
    for norm in all_norm:
        originals = set()
        for m in normalized_map.values():
            if norm in m:
                originals.add(m[norm])
        if len(originals) > 1:
            potential_mismatches.append((norm, sorted(originals)))

missing_by_dataset, potential_mismatches

In [ ]:
report_lines = []
report_lines.append('Raw Data Inspection Report')
report_lines.append('==========================')
report_lines.append('')

for name, summary in summaries.items():
    report_lines.append(f'Dataset: {name}')
    report_lines.append(f